## 1 — Ollama + gpt-oss-20b

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"
env["OLLAMA_ORIGINS"] = "*"

import subprocess
def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"], env=env)

import threading
thread = threading.Thread(target=run_ollama_serve)
thread.start()

import time
time.sleep(5)

In [ ]:
!ollama pull gpt-oss:20b
!curl http://localhost:11434/api/generate -d '{ "model": "gpt-oss:20b", "prompt": "Who are you?", "stream": false}'

In [ ]:
# TODO: Chạy lệnh dưới đây ở terminal để tunnel ra bên ngoài CHÚ Ý: NHẤN PHẢI RỒI CHỌN COPY TRONG MENU, NẾU BẤM CTRL + C SẼ TẮT KẾT NỐI
# ssh -p 443 -R0:localhost:11434 qr@a.pinggy.io

In [ ]:
import requests

url = "public_pinggy_port/api/chat"

payload = {
    "model": "gpt-oss:20b",
    "messages": [
        {"role": "user", "content": "Trí tuệ nhân tạo là gì? Trả lời trong 2 câu."}
    ],
    "stream": False
}

response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()

data = response.json()
print(data["message"]["content"])

## 2 — Intent Classifier Server (Lab 2 LoRA adapter)

This section loads your Lab 2 checkpoint from Hugging Face and exposes a `/predict` endpoint.

- Repo id for code: `letrungtin2k5/lab2-artifacts`

The notebook will download the repo snapshot, read `lab2-artifacts/inference.yaml`, resolve the adapter path, and run inference the same way as Lab 2 `inference.py`.

In [ ]:
# Install dependencies for the intent inference server
!pip install -q fastapi uvicorn nest-asyncio peft transformers accelerate
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import torch
import json
import yaml
from pathlib import Path
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

nest_asyncio.apply()

INTENT_REPO_ID = "letrungtin2k5/lab2-artifacts"
INTENT_REPO_DIR = "lab2-artifacts"
INTENT_PORT = 8000

repo_root = Path(snapshot_download(repo_id=INTENT_REPO_ID, repo_type="model"))
repo_base = repo_root / INTENT_REPO_DIR

def resolve_path(path_str: str) -> Path:
    candidate = Path(path_str)
    for item in (candidate, repo_root / candidate, repo_base / candidate):
        if item.exists():
            return item.resolve()
    return (repo_root / candidate).resolve()

with open(resolve_path("lab2-artifacts/inference.yaml"), "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

adapter_path = resolve_path(cfg["adapter_path"])
used_label_map_name = Path(cfg.get("used_label_text_map", "used_label_text_map.json")).name
used_label_map_path = adapter_path / used_label_map_name
if not used_label_map_path.exists():
    used_label_map_path = resolve_path(cfg.get("used_label_text_map", used_label_map_name))
    if not used_label_map_path.exists():
        used_label_map_path = resolve_path(cfg["label_text_map"])

with open(used_label_map_path, "r", encoding="utf-8") as f:
    id2label_raw = json.load(f)
valid_labels = set(id2label_raw.values())

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(adapter_path),
    max_seq_length=cfg["max_seq_length"],
    dtype=cfg.get("dtype", None),
    load_in_4bit=cfg.get("load_in_4bit", True),
)
if getattr(model, "generation_config", None) is not None:
    model.generation_config.max_length = None
tokenizer = get_chat_template(tokenizer, chat_template=cfg["chat_template"])
FastLanguageModel.for_inference(model)
model.eval()

print(f"Loaded {len(valid_labels)} labels from {used_label_map_path.name}")


def build_prompt(message: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": f"Classify the intent: {message}"}],
        tokenize=False,
        add_generation_prompt=True,
    )


def normalize_prediction(predicted_text: str) -> str:
    predicted_label = predicted_text.strip().split("\n")[0].strip()
    if predicted_label in valid_labels:
        return predicted_label
    lut = {x.lower(): x for x in valid_labels}
    return lut.get(predicted_label.lower(), "unknown_intent")


intent_app = FastAPI(title="Intent Classifier")


class IntentRequest(BaseModel):
    message: str


@intent_app.post("/predict")
def predict(req: IntentRequest):
    inputs = tokenizer(build_prompt(req.message), return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=cfg["max_new_tokens"],
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    predicted_text = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    # The Lab 2 classifier does not expose a calibrated confidence score.
    return {"intent": normalize_prediction(predicted_text), "confidence": None}


@intent_app.get("/health")
def health():
    return {"status": "ok"}


def run_intent_server():
    uvicorn.run(intent_app, host="0.0.0.0", port=INTENT_PORT)


threading.Thread(target=run_intent_server, daemon=True).start()
import time; time.sleep(3)
print(f"Intent server running on http://localhost:{INTENT_PORT}")

In [ ]:
# Quick smoke-test against the local server before tunnelling
import requests, json

resp = requests.post(
    "http://localhost:8000/predict",
    json={"message": "I made a transfer 3 days ago but the recipient has not received the money."},
    timeout=60,
)
print(json.dumps(resp.json(), indent=2))

## Expose the Intent Classifier with Pinggy

Run this command in a Colab terminal after the intent server is running. Copy the generated public URL and use it as `INTENT_API_URL` with `/predict` appended, for example `http://yyyy.a.free.pinggy.link/predict`.

In [ ]:
# Tunnel the intent classifier server on port 8000
# ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io